In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/insurance_data_cleaned.csv"
)

C:\Users\mijuu\AppData\Local\Temp\ipykernel_22664\2515150779.py:4: DtypeWarning: Columns (0: mmcode, 1: Cylinders, 2: cubiccapacity, 3: kilowatts, 4: NumberOfDoors, 5: CapitalOutstanding) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


MODEL 1 — Claim Severity Prediction

In [ ]:
df_claims = df[df['TotalClaims'] > 0]

MODEL 2 — Claim Probability

In [ ]:
df['HasClaim'] = np.where(
    df['TotalClaims'] > 0,
    1,
    0
)

Feature Engineering

In [2]:
df['VehicleAge'] = (
    2015 - df['RegistrationYear']
)

In [3]:
df['LossRatio'] = np.where(
    df['TotalPremium'] == 0,
    0,
    df['TotalClaims'] / df['TotalPremium']
)

Select Features

In [4]:
features = [
    'Province',
    'VehicleType',
    'make',
    'Gender',
    'CustomValueEstimate',
    'SumInsured',
    'kilowatts',
    'VehicleAge'
]

In [5]:
target = 'TotalClaims'

In [6]:
#Handle Missing Values

num_cols = [
    'CustomValueEstimate',
    'SumInsured',
    'kilowatts',
    'VehicleAge'
]

for col in num_cols:
    df[col] = df[col].fillna(
        df[col].median()
    )

TypeError: Cannot perform reduction 'median' with string dtype

In [7]:
cat_cols = [
    'Province',
    'VehicleType',
    'make',
    'Gender'
]

for col in cat_cols:
    df[col] = df[col].fillna(
        'Unknown'
    )

In [8]:
#Encode Categorical Variables

df_encoded = pd.get_dummies(
    df[features],
    drop_first=True
)

Train/Test Split

In [9]:
from sklearn.model_selection import train_test_split

X = df_encoded
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Linear Regression

In [10]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

In [11]:
from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

rmse_lr = np.sqrt(
    mean_squared_error(
        y_test,
        pred_lr
    )
)

r2_lr = r2_score(
    y_test,
    pred_lr
)

print("RMSE:", rmse_lr)
print("R2:", r2_lr)

RMSE: 2213.200281862634
R2: -0.0024288704641903802


Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

In [ ]:
rmse_rf = np.sqrt(
    mean_squared_error(
        y_test,
        pred_rf
    )
)

r2_rf = r2_score(
    y_test,
    pred_rf
)

print(rmse_rf)
print(r2_rf)

XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)

In [ ]:
rmse_xgb = np.sqrt(
    mean_squared_error(
        y_test,
        pred_xgb
    )
)

r2_xgb = r2_score(
    y_test,
    pred_xgb
)

print(rmse_xgb)
print(r2_xgb)

Compare Models

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Random Forest',
        'XGBoost'
    ],
    
    'RMSE': [
        rmse_lr,
        rmse_rf,
        rmse_xgb
    ],
    
    'R2': [
        r2_lr,
        r2_rf,
        r2_xgb
    ]
})

results

SHAP Explainer

In [ ]:
import shap

explainer = shap.TreeExplainer(rf)

shap_values = explainer.shap_values(X_test)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test
)